## Reading in the raw file format (OBF, .sec)

In [ ]:
pip install msr-reader

In [ ]:
from msr_reader import OBFFile
import glob
import imageio as io
import numpy as np
import cv2

img_folder = "test_images"
sec_files = glob.glob("{dir}/*.sec".format(dir=img_folder))
sec_file = sec_files[0]
print(sec_file)


def normalized_float2int8(img):
    return (255 * (img - img.min()) / (img.ptp() + 1e-8)).astype(np.uint8)


def laplacian_var(img):
    laplacian = cv2.Laplacian(img, cv2.CV_64F)
    return laplacian.var()


def sharpness(img):
    sharpness_vals = []
    for i in range(img.shape[0]):
        z_img = img[i]
        if z_img.dtype != np.uint8:
            z_img = normalized_float2int8(z_img)
        lap_var = laplacian_var(z_img)
        sharpness_vals.append(lap_var)
    return sum(sharpness_vals) / len(sharpness_vals)


with OBFFile(sec_file) as f:
    max_sharpness = 0
    max_sharpness_idx = 0

    # reading image data
    for idx in range(f.num_stacks):
        img = f.read_stack(idx)  # read stack with index idx into numpy array

        # Sharpness
        img_sharpness = sharpness(img)
        if img_sharpness > max_sharpness:
            max_sharpness = img_sharpness
            max_sharpness_idx = idx

        # metadata
        stack_shapes = f.shapes  # list of stack shapes, including stack and dimension names
        # like shapes, but with pixel sizes (unit: meters)
        pixel_sizes = f.pixel_sizes
        print(stack_shapes)
        print(pixel_sizes)

    # write sharpest image
    io.mimwrite(f"{img_folder}/out/s2023-10-27_m157_yxz_100x_biocytin_ATTO647N_orig.tif",
                f.read_stack(max_sharpness_idx))
    stacks_metadata = {'pixel_sizes': f.pixel_sizes, 'stack_shapes': f.shapes}

## Reading in the masks (.mat)

In [ ]:
import flammkuchen as fl

mat_files = glob.glob("{dir}/*.mat".format(dir=img_folder))
print(mat_files)
masks = list()

for mat_file in mat_files:
    masks.append(fl.load(mat_file))

In [ ]:
for i, m in enumerate(masks):
    print(f"Item {i}: type = {type(m)}")
    print("Keys:", list(m.keys()))
    for k, v in m.items():
        print(f"  {k}: type = {type(v)}, shape = {getattr(v, 'shape', 'N/A')}")

# Each mask element has:
# - #refs#
# - mask
# Each of those has a type and a shape:
# - #refs#: type dictionary, no shape
# - mask: type numpy array, shape (rows, cols)

### Plotting the relevant ones

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

mask = masks[0]['mask']
print(mask.shape)
# mask has shape (36, 1), meaning that it has 36 rows of 1 element each

# We want to overlay all the masks in a same stack. We start with the very first mask, and
# put the rest on top of it with different values (i.e. binary masks converted from 0/1 to 0/idx)
idx = 0
stack = np.zeros(mask[0][0].shape, dtype=np.uint8)

# Iterate in rows
for i in range(mask.shape[0]):
    # print(mask[i][0].shape)
    # The shape should be (17, 520, 1630), meaning that it's a 17x520x1630 (OZxOYxOX) 3D binary mask
    if len(mask[i][0].shape) == 3:
        # plt.figure()
        # plt.imshow(mask[i][0].max(0))
        # plt.colorbar()
        idx += 1
        # Put mask into stack with 'idx' where originally there were 1
        stack[mask[i][0] > 0] = idx

In [ ]:
plt.imshow(stack.max(0), cmap='turbo')
plt.colorbar()

### Saving a stack as tif file

In [ ]:
import imageio as io
io.mimwrite(
    f"{img_folder}/out/2023-10-27_m157_yxz_100x_biocytin_ATTO647N_stack.tif", stack)

## Semantic segmentation

In [ ]:
max_pixel_count = 0
dendrite_idx = stack.max()
# Find dendrite mask, ignore background
i = 1
while i <= mask.shape[0]:
    pixel_count = np.sum(stack == i)
    if pixel_count > max_pixel_count:
        max_pixel_count = pixel_count
        dendrite_idx = i
    i += 1
print(f"Mask {dendrite_idx} is most likely the dendrite, with {max_pixel_count} active pixels")

spines = ((stack > 0) & (stack != dendrite_idx)).astype(np.uint8) * 255
dendrite = (stack == dendrite_idx).astype(np.uint8) * 255

plt.subplot(121)
plt.imshow(spines.max(0))

plt.subplot(122)
plt.imshow(dendrite.max(0))

io.mimwrite(
    f"{img_folder}/out/2023-10-27_m157_yxz_100x_biocytin_ATTO647N_spines.tif", spines)
io.mimwrite(
    f"{img_folder}/out/2023-10-27_m157_yxz_100x_biocytin_ATTO647N_dendrite.tif", dendrite)

## Test model

### Set up path to DeepD3 project

In [ ]:
import sys
import IPython
from pathlib import Path

current_path = "/".join(
    IPython.extract_module_locals()[1]["__vsc_ipynb_file__"].split("/")[-5:]
)
deepd3_root_path = Path(current_path).resolve().parent.parent
sys.path.insert(0, f"{deepd3_root_path}")
sys.path.insert(0, f"{deepd3_root_path}/deepd3")

### Load training data

In [ ]:
from deepd3.training.tile import TiledDataGenerator

TRAINING_DATA_PATH = "test_images/out/test.d3set"

dg_training = TiledDataGenerator(fn=TRAINING_DATA_PATH,
                                 batch_size=64,  # Data processed at once, depends on your GPU
                                 target_resolution=0.02,  # fixed to 20 nm, can be None for mixed resolution training
                                 # 128x4= 512 because we have 4x more resolution
                                 size=(1, 512, 512)
                                 ).get_batch(0)
dg_validation = dg_training  # temporarily use training data as validation

### Visualize training data

In [ ]:
X, Y = dg_training

for i in range(64):
    plt.figure(figsize=(12, 4))

    plt.subplot(131)
    plt.imshow(X[i].squeeze(), cmap='gray')
    plt.colorbar()

    plt.subplot(132)
    plt.imshow(Y[0][i].squeeze(), cmap='gray')
    plt.colorbar()

    plt.subplot(133)
    plt.imshow(Y[1][i].squeeze(), cmap='gray')
    plt.colorbar()

plt.tight_layout()

### Compile model

In [ ]:
from deepd3.model import DeepD3_Model
import segmentation_models as sm
from tensorflow.keras.optimizers import Adam
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"
### ROCm dependencies incantations ###
os.environ["LD_LIBRARY_PATH"] = "/opt/rocm/lib"
# Force preload
# import ctypes
# ctypes.CDLL("/opt/rocm/lib/librccl.so.1")
######################################
sm.set_framework("tf.keras")


# Create a naive DeepD3 model with a given base filter count (e.g. 32)
model = DeepD3_Model(filters=32)

# Set appropriate training settings
model.compile(Adam(learning_rate=0.0005),  # optimizer, good default setting, can be tuned
              # Dice loss for dendrite, MSE for spines
              [sm.losses.dice_loss, "mse"],
              metrics=['acc', sm.metrics.iou_score])  # Metrics for monitoring progress

model.summary()

### Fitting model

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler


def schedule(epoch, lr):
    if epoch < 15:
        return lr

    else:
        return lr * tf.math.exp(-0.1)

### Training model

In [ ]:
EPOCHS = 1

# Save best model automatically during training
mc = ModelCheckpoint("DeepD3_test_model.h5",
                     save_best_only=True)

# Save metrics
csv = CSVLogger("DeepD3_test_model.csv")

# Adjust learning rate during training to allow for better convergence
lrs = LearningRateScheduler(schedule)

# Actually train the network
h = model.fit(dg_training,
              batch_size=32,
              epochs=EPOCHS,
              validation_data=dg_validation,
              callbacks=[mc, csv, lrs])